In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [3]:
df = pd.read_csv('../data/raw/creditcard.csv')

In [4]:
y = df['Class']

In [5]:
X = df.drop(columns =['Class'])

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,              
    y,            
    test_size=0.2, 
    random_state=42,
    stratify= y
)

In [7]:
forest = RandomForestClassifier(n_jobs = -1, max_depth = 7)

In [8]:
forest

,n_estimators,100
,criterion,'gini'
,max_depth,7
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
forest.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,7
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
predictt = forest.predict(X_test)

In [11]:
predictt

array([0, 0, 0, ..., 0, 0, 0], shape=(56962,))

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score
)

In [13]:
def check_threshold(y_true, y_proba, threshold=0.5):

    y_pred = (y_proba >= threshold).astype(int)
    
    # Считаем метрики
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"\n🔹 ПОРОГ: {threshold}")
    print("-" * 30)
    print(f"Precision: {precision:.4f} (сколько из предсказанных fraud - реальные)")
    print(f"Recall:    {recall:.4f} (сколько реальных fraud поймали)")
    print(f"F1-score:  {f1:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"\nConfusion Matrix:")
    print(f"TN: {tn} | FP: {fp}")
    print(f"FN: {fn} | TP: {tp}")
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
    }

y_proba = forest.predict_proba(X_test)[:, 1] 

In [16]:
check_threshold(y_test, y_proba, threshold=0.5)


🔹 ПОРОГ: 0.5
------------------------------
Precision: 0.9620 (сколько из предсказанных fraud - реальные)
Recall:    0.7755 (сколько реальных fraud поймали)
F1-score:  0.8588
Accuracy:  0.9996

Confusion Matrix:
TN: 56861 | FP: 3
FN: 22 | TP: 76


{'precision': 0.9620253164556962,
 'recall': 0.7755102040816326,
 'f1': 0.8587570621468926,
 'accuracy': 0.9995611109160493,
 'tn': np.int64(56861),
 'fp': np.int64(3),
 'fn': np.int64(22),
 'tp': np.int64(76)}

In [15]:
importances = forest.feature_importances_

# Создаем DataFrame для красивого вывода
feature_importance = pd.DataFrame({
    'feature': X_train.columns,  # или df.drop('Class', axis=1).columns
    'importance': importances
}).sort_values('importance', ascending=False)

# Простой вывод
print(feature_importance)

# Или только топ-10
print("\nТОП-10 САМЫХ ВАЖНЫХ ПРИЗНАКОВ:")
print(feature_importance.head(10))

# Или в процентах
feature_importance['importance_percent'] = feature_importance['importance'] * 100
print("\nВ ПРОЦЕНТАХ:")
print(feature_importance.head(10))

   feature  importance
12     V12    0.165606
14     V14    0.162197
17     V17    0.132701
10     V10    0.104015
11     V11    0.082466
16     V16    0.060575
18     V18    0.043845
9       V9    0.041484
4       V4    0.025933
7       V7    0.023856
21     V21    0.016516
26     V26    0.014019
8       V8    0.012176
2       V2    0.010564
6       V6    0.010212
27     V27    0.010194
20     V20    0.009960
1       V1    0.008796
5       V5    0.008663
3       V3    0.006917
28     V28    0.006702
19     V19    0.006170
15     V15    0.006049
22     V22    0.006028
0     Time    0.005955
24     V24    0.004744
25     V25    0.004484
29  Amount    0.004273
13     V13    0.003003
23     V23    0.001898

ТОП-10 САМЫХ ВАЖНЫХ ПРИЗНАКОВ:
   feature  importance
12     V12    0.165606
14     V14    0.162197
17     V17    0.132701
10     V10    0.104015
11     V11    0.082466
16     V16    0.060575
18     V18    0.043845
9       V9    0.041484
4       V4    0.025933
7       V7    0.023856

В